In [3]:
import joblib
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Load dataset
df = pd.read_csv("vaers_sample_cleaned.csv")

# Target variable
y = df["any_serious"].apply(lambda x: 1 if x == "Yes" else 0).values

# TF-IDF on symptom text
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["symptoms_normalized"].fillna(""))

# Save sparse features (optional)
sparse.save_npz("features_sparse.npz", X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train logistic regression
clf = LogisticRegression(class_weight="balanced", max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

# Save model + vectorizer
joblib.dump(clf, "logreg_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("✅ Model and vectorizer saved")


              precision    recall  f1-score   support

           0       0.99      0.88      0.93    233982
           1       0.26      0.85      0.40     11069

    accuracy                           0.88    245051
   macro avg       0.62      0.87      0.67    245051
weighted avg       0.96      0.88      0.91    245051

ROC-AUC: 0.9432830107235741
✅ Model and vectorizer saved
